# Global error-budget allocation for the adaptive block mixed-precision Cholesky factorization

The notebook `jcf-gp-knee-xyz-colab.ipynb` fits a trivariate Gaussian process emulator of the three components of the knee joint contact force with the adaptive block mixed-precision Cholesky factorization of Salvaña (2026). That factorization stores each off-diagonal block of the Cholesky factor in the lowest precision that a per-block error budget allows, and the budget is the same for every block. This notebook asks whether a global error budget, shared across the blocks, stores more of the factor in low precision while keeping the same positive-definiteness guarantee.

The test problem is the trivariate separable space-time Matérn emulator of the knee contact force components over strides and gait points, whose covariance matrix has $N = 3 \times n \times 101$ rows for $n$ training strides and no Kronecker structure. The parameters are estimated once with the original rule and then held fixed, so that every method factors the same matrix and the differences come from the precision-assignment rule alone. Four rules are compared: the original per-block rule and three allocations of the global budget, a greedy one, a paced one and an offline optimal one. For each, the notebook reports the precision mix of the factor blocks, the stored size of the factor, the realized perturbation of the covariance matrix against its guaranteed bound, the factorization time and the effect on the likelihood and the predictions.

The notebook has four parts. It loads the data, it defines the model and the four rules with the theory behind them, it fits the parameters and runs the comparison, and it reports the results in tables and figures. It runs on any GPU runtime. In Colab, choose Runtime, then Change runtime type, then a GPU.

## Setup

In [ ]:
import gc, json, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import minimize
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

assert torch.cuda.is_available(), "No GPU is attached. Choose Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{GPU_NAME}, {GPU_GB:.1f} GB")

## Data

The data are those of the first notebook. One row is one stride of the right leg, and there are 480 training strides from twelve participants and 140 test strides from three other participants. The inputs are three ground reaction force curves, five joint angle curves, the belt speed, the body mass and the height. The outputs used here are the three components of the right knee joint contact force computed by OpenSim, in body weights (BW), time normalized to 101 points of the gait cycle: $f_x$ is anterior-posterior (shear), $f_y$ is along the long axis of the tibia and negative in compression, and $f_z$ is medio-lateral.

The size of the comparison is set by `STRIDES`, the number of training strides, evenly spaced over the 480, and it is the only value that has to change to go from a small test to the full data. With 24 strides the matrix has $N = 7{,}272$ rows and the whole notebook runs in a few minutes on any GPU. With 480 strides it has $N = 145{,}440$ rows, the factor stored by the per-block rule is about 42 GB, and the comparison needs the 80 GB NVIDIA A100 and about half an hour. The stored factor grows with the square of `STRIDES` and the factorization time with its cube. The parameters are fitted on the same strides, so that each rule is estimated and then factored on one matrix. For the full data, where four fits of the 145,440 matrix would take days, `FIT_STRIDES` can be set to a smaller number to fit on a subset of the comparison strides and compare at those estimates.

In [ ]:
!gdown --folder "https://drive.google.com/drive/folders/1CI5OAxzB9qYI9vIxW1f0cGE4SN1hhyBj" -q

In [ ]:
STRIDES = 24                                               # training strides of the comparison; N = 3 x STRIDES x 101; 480 for the full data
FIT_STRIDES = None                                         # None: the parameters are fitted on the same strides as the comparison; a smaller number fits on a subset of them, for the full data
TEST_STRIDES = 8                                           # test strides used for the prediction check
JOINT = "knee"                                             # "hip", "knee" or "ankle"
COMPONENTS = ["fx", "fy", "fz"]

x_train = pd.read_csv("data/x-train.csv")
y_train = pd.read_csv("data/y-train.csv")
x_test = pd.read_csv("data/x-test.csv").iloc[:TEST_STRIDES].reset_index(drop=True)
y_test = pd.read_csv("data/y-test.csv").iloc[:TEST_STRIDES].reset_index(drop=True)
keep = np.unique(np.linspace(0, len(x_train) - 1, STRIDES).round().astype(int))
x_train, y_train = x_train.iloc[keep].reset_index(drop=True), y_train.iloc[keep].reset_index(drop=True)
x_train.shape, y_train.shape, x_test.shape, y_test.shape

Each stride becomes the vector $\mathbf{x}_i$ of its eight input curves and three scalars, standardized on the training strides exactly as in the first notebook. The Gaussian process models the residual of each component curve, the curve minus the mean curve $m_j(t)$ of the training strides, divided by the standard deviation $s_j$ of the training curves of that component.

In [ ]:
GRID = np.arange(101)
INPUT_CURVES = ["grf_vertical_BW", "grf_anteroposterior_BW", "grf_mediolateral_BW",
                "hip_flexion_deg", "hip_adduction_deg", "hip_rotation_deg", "knee_flexion_deg", "ankle_angle_deg"]

def curve(df, name):
    return df[[f"{name}_t{t:03d}" for t in GRID]].to_numpy()

def build_features(x, x_ref):
    cols = []
    for name in INPUT_CURVES:
        C, C_ref = curve(x, name), curve(x_ref, name)
        cols.extend(((C - C_ref.mean(axis=0)) / C_ref.std()).T)
    for name in ["belt_speed_mps", "mass_kg", "height_m"]:
        cols.append((x[name].to_numpy() - x_ref[name].mean()) / x_ref[name].std())
    return np.array(cols).T

F_train = build_features(x_train, x_train)
F_test = build_features(x_test, x_train)
Y_train = np.stack([curve(y_train, f"{JOINT}_jcf_{c}_BW") for c in COMPONENTS])      # (3 components, strides, 101 gait points), in BW
Y_test = np.stack([curve(y_test, f"{JOINT}_jcf_{c}_BW") for c in COMPONENTS])
mean_t = Y_train.mean(axis=1)                                                           # mean curve of each component
scale = Y_train.std(axis=(1, 2))                                                        # s_j, in BW
Z_train = (Y_train - mean_t[:, None, :]) / scale[:, None, None]                         # standardized residuals
fit_rows = np.arange(len(keep)) if FIT_STRIDES is None else np.unique(np.linspace(0, len(keep) - 1, min(FIT_STRIDES, len(keep))).round().astype(int))   # the fit strides
F_fit, Z_fit = F_train[fit_rows], Z_train[:, fit_rows, :]
Z_train.shape, Z_fit.shape

## Trivariate separable space-time Matérn model

Let $i = 1, \dots, n$ index the training strides, let $t \in \{0, 1, \dots, 100\}$ denote the percent of the gait cycle, and let $j \in \{1, 2, 3\}$ index the anterior-posterior, the axial and the medio-lateral component. The input of stride $i$ is the vector $\mathbf{x}_i \in \mathbb{R}^{811}$ and $y_{ij}(t)$ is component $j$ at gait point $t$ of stride $i$. Each component follows
$$
y_{ij}(t) = m_j(t) + s_j \big( f_j(\mathbf{x}_i, t) + \varepsilon_{ij}(t) \big), \qquad \varepsilon_{ij}(t) \sim \mathcal{N}(0, \sigma_{\varepsilon, j}^2),
$$
where $(f_1, f_2, f_3)$ is a trivariate Gaussian process. Write $h$ for the root mean squared difference between the standardized inputs of two strides and $u = |t - t'|$ for the lag between two gait points. The covariance function is
$$
\operatorname{Cov}\!\big(f_j(\mathbf{x}, t), f_k(\mathbf{x}', t')\big) = \beta_{jk}\, \sigma_j \sigma_k \;\, \gamma_{jk}\, \mathcal{M}\!\Big(\frac{h}{\ell_{jk}}\Big) \;\, \gamma_{t, jk}\, \mathcal{M}\!\Big(\frac{u}{\ell_{t, jk}}\Big), \qquad \mathcal{M}(d) = \Big(1 + \sqrt{5}\, d + \tfrac{5}{3} d^2\Big) e^{-\sqrt{5}\, d},
$$
with $\mathcal{M}$ the Matérn correlation function with smoothness $5/2$. Component $j$ has its own variance $\sigma_j^2$, input range $\ell_j$ and temporal range $\ell_{t, j}$, the parameters $\beta_{jk}$ form a correlation matrix, and for a pair of components
$$
\frac{1}{\ell_{jk}^2} = \frac{1}{2} \Big( \frac{1}{\ell_j^2} + \frac{1}{\ell_k^2} \Big), \qquad \gamma_{jk} = \Big( \frac{2\, \ell_j \ell_k}{\ell_j^2 + \ell_k^2} \Big)^{5/2},
$$
and likewise for the temporal ranges (Gneiting, Kleiber and Schlather, 2010; Apanasovich, Genton and Sun, 2012). The cross-correlation between components $j$ and $k$ at the same stride and gait point is $\rho_{jk} = \beta_{jk}\, \gamma_{jk}\, \gamma_{t, jk}$.

The model has fourteen parameters. Writing $\boldsymbol{\Sigma} = \sigma_1^2 \bar{\boldsymbol{\Sigma}}$, the variance of the first component is profiled out, $\hat{\sigma}_1^2 = \mathbf{y}^\top \bar{\boldsymbol{\Sigma}}^{-1} \mathbf{y} / N$, and the negative log likelihood of the stacked residuals $\mathbf{y}$ is
$$
-\log p = \tfrac{1}{2} \Big( N \log \hat{\sigma}_1^2 + \log \det \bar{\boldsymbol{\Sigma}} + N + N \log 2\pi \Big).
$$
The matrix $\bar{\boldsymbol{\Sigma}}$ is a nonnegative definite matrix plus the diagonal matrix of the relative nugget effects $\eta_j = \sigma_{\varepsilon, j}^2 / \sigma_1^2$, so $m = \min_j \eta_j$ is a lower bound on its smallest eigenvalue, available at no cost. The class below orders the points by component, then stride, then gait point, and returns any block of $\bar{\boldsymbol{\Sigma}}$ on the GPU in double precision, so the full matrix is never stored.

In [ ]:
def matern(d):
    s = 5.0**0.5 * d
    return (1.0 + s + s * s / 3.0) * torch.exp(-s)


def correlation_matrix(raw):
    # 3 x 3 correlation matrix from three unconstrained numbers, through the angles of its Cholesky factor
    a = (np.pi / 2) * np.tanh(np.asarray(raw, dtype=float))
    L = np.array([[1.0, 0.0, 0.0],
                  [np.sin(a[0]), np.cos(a[0]), 0.0],
                  [np.sin(a[1]), np.cos(a[1]) * np.sin(a[2]), np.cos(a[1]) * np.cos(a[2])]])
    return L @ L.T


def pair_ranges(l):
    # 1 / l_jk and gamma_jk for every pair of components
    a2 = 1.0 / l**2
    inverse = np.sqrt((a2[:, None] + a2[None, :]) / 2)
    return inverse, (np.sqrt(a2[:, None] * a2[None, :]) / inverse**2) ** 2.5


class SpaceTimeTrivariateMatern:
    def __init__(self, F, Z, t, block_size):
        self.n, self.T, self.N = Z.shape[1], Z.shape[2], Z.size
        Ft = torch.tensor(F, dtype=torch.float64, device=DEVICE)
        self.F = Ft
        self.H = torch.cdist(Ft, Ft, compute_mode="donot_use_mm_for_euclid_dist") / F.shape[1]**0.5      # h between strides
        self.t = torch.tensor(t, dtype=torch.float64, device=DEVICE)
        self.y = torch.tensor(Z.reshape(-1), dtype=torch.float64, device=DEVICE)                          # stacked residuals
        index = torch.arange(self.N, device=DEVICE)
        self.component, self.stride, self.time = index // (self.n * self.T), (index // self.T) % self.n, index % self.T
        self.blocks = [slice(a, min(a + block_size, self.N)) for a in range(0, self.N, block_size)]

    def set_parameters(self, theta):
        # theta = log input ranges (3), log temporal ranges (3), correlation angles (3), log variance ratios of fy and fz (2), log relative nugget effects (3)
        l, lt = np.exp(theta[0:3]), np.exp(theta[3:6])
        beta = correlation_matrix(theta[6:9])
        v = np.concatenate([[1.0], np.exp(theta[9:11])])
        eta = np.exp(theta[11:14])
        inverse_l, gamma = pair_ranges(l)
        inverse_lt, gamma_t = pair_ranges(lt)
        rho = beta * gamma * gamma_t                                               # cross-correlations at the same stride and gait point
        as_tensor = lambda a: torch.tensor(a, dtype=torch.float64, device=DEVICE)
        self.amplitude = as_tensor(rho * np.sqrt(np.outer(v, v)))
        self.inverse_l, self.inverse_lt = as_tensor(inverse_l), as_tensor(inverse_lt)
        self.nugget = as_tensor(eta)[self.component]                               # relative nugget effect of every point
        self.m = float(eta.min())                                                  # lower bound on the smallest eigenvalue
        self.parameters = dict(l=l, lt=lt, v=v, eta=eta, rho=rho)

    def block(self, rows, cols):
        # block of the covariance matrix between two sets of points, without the nugget effect
        pair = (self.component[rows][:, None], self.component[cols][None, :])
        h = self.H[self.stride[rows]][:, self.stride[cols]]
        u = (self.t[self.time[rows]][:, None] - self.t[self.time[cols]][None, :]).abs()
        return self.amplitude[pair] * matern(h * self.inverse_l[pair]) * matern(u * self.inverse_lt[pair])

    def cross(self, F_new, rows_train):
        # covariance between every (component, new stride, gait point) and the training points in rows_train, without the nugget effect
        Fn = torch.tensor(F_new, dtype=torch.float64, device=DEVICE)
        h_new = torch.cdist(Fn, self.F, compute_mode="donot_use_mm_for_euclid_dist") / self.F.shape[1]**0.5    # (new strides, training strides)
        n_new = Fn.shape[0]
        idx = torch.arange(3 * n_new * self.T, device=DEVICE)
        c, s, t = idx // (n_new * self.T), (idx // self.T) % n_new, idx % self.T
        pair = (c[:, None], self.component[rows_train][None, :])
        h = h_new[s][:, self.stride[rows_train]]
        u = (self.t[t][:, None] - self.t[self.time[rows_train]][None, :]).abs()
        return self.amplitude[pair] * matern(h * self.inverse_l[pair]) * matern(u * self.inverse_lt[pair])

## Adaptive block mixed-precision Cholesky factorization and the two error budgets

### The storage perturbation

Partition $\bar{\boldsymbol{\Sigma}}$ into an $n_B \times n_B$ grid of blocks. The factorization proceeds by block columns. All arithmetic is in `fp64`, the diagonal factor blocks are kept in `fp64`,
$$
\widetilde{\mathbf{L}}_{kk} = \operatorname{chol}\Big( \bar{\boldsymbol{\Sigma}}_{kk} - \sum_{j < k} \widetilde{\mathbf{L}}_{kj} \widetilde{\mathbf{L}}_{kj}^\top \Big),
$$
and each off-diagonal block is computed in `fp64` as
$$
\mathbf{A}_{ik} = \Big( \bar{\boldsymbol{\Sigma}}_{ik} - \sum_{j < k} \widetilde{\mathbf{L}}_{ij} \widetilde{\mathbf{L}}_{kj}^\top \Big) \widetilde{\mathbf{L}}_{kk}^{-\top}
$$
and then stored in some format $p$, $\widetilde{\mathbf{L}}_{ik} = \mathrm{fl}_p(\mathbf{A}_{ik})$, with conversion error $\mathbf{E}_{ik} = \mathrm{fl}_p(\mathbf{A}_{ik}) - \mathbf{A}_{ik}$. The covariance matrix implied by the stored factor is $\widetilde{\boldsymbol{\Sigma}} = \widetilde{\mathbf{L}} \widetilde{\mathbf{L}}^\top$, and the two displays above give the structure of the perturbation $\boldsymbol{\Delta} = \widetilde{\boldsymbol{\Sigma}} - \bar{\boldsymbol{\Sigma}}$ exactly, under exact `fp64` arithmetic: its diagonal blocks are zero, because the diagonal factor blocks reproduce $\bar{\boldsymbol{\Sigma}}_{kk}$ whatever was stored before, and its off-diagonal blocks are
$$
\boldsymbol{\Delta}_{ik} = \mathbf{E}_{ik} \widetilde{\mathbf{L}}_{kk}^\top, \qquad i > k.
$$
Hence
$$
\|\boldsymbol{\Delta}\|_F^2 = 2 \sum_{k=1}^{n_B} \sum_{i=k+1}^{n_B} \big\| \mathbf{E}_{ik} \widetilde{\mathbf{L}}_{kk}^\top \big\|_F^2 .
$$
By Weyl's inequality, $\lambda_{\min}(\widetilde{\boldsymbol{\Sigma}}) \ge \lambda_{\min}(\bar{\boldsymbol{\Sigma}}) - \|\boldsymbol{\Delta}\|_2 \ge m - \|\boldsymbol{\Delta}\|_F$, so $\widetilde{\boldsymbol{\Sigma}}$ is positive definite whenever $\|\boldsymbol{\Delta}\|_F \le \tau m$ for some $0 < \tau < 1$. The same bound, applied to the partially stored matrix at every block column, is what makes every diagonal Cholesky exist, so the factorization cannot break down. Every rule below is a way of keeping $\|\boldsymbol{\Delta}\|_F \le \tau m$ while storing as many blocks as possible in low precision. The storage formats are `fp8`, the 8-bit format e4m3 with 1 byte per entry, `bf16` with 2, `fp32` with 4 and `fp64` with 8. All of them are storage formats only: a stored block is read back in `fp64` before it enters any product, so the method needs no low-precision arithmetic, and `fp8` storage is available on any CUDA GPU, including the A100 used in the paper, although `fp8` arithmetic units exist only from the H100 on. Conversion to `fp8` overflows above 448, and the a posteriori test below measures the actual conversion error, so subnormal and overflowing entries are handled by the measurement rather than by an assumption.

### Method A, the per-block budget of Salvaña (2026)

The original rule gives every off-diagonal block the same share of the budget. Block $(i, k)$ is stored in the lowest format $p$, in the order `fp8`, `bf16`, `fp32`, that satisfies
$$
\big\| \mathrm{fl}_p(\mathbf{A}_{ik}) - \mathbf{A}_{ik} \big\|_F \, \big\| \widetilde{\mathbf{L}}_{kk} \big\|_2 \le b = \frac{\tau m}{n_B},
$$
and in `fp64` if none does. Since $\|\boldsymbol{\Delta}_{ik}\|_F \le \|\mathbf{E}_{ik}\|_F \|\widetilde{\mathbf{L}}_{kk}\|_2 \le b$ for every block and there are fewer than $n_B^2$ blocks including their transposes, $\|\boldsymbol{\Delta}\|_F \le n_B\, b = \tau m$. This is Proposition 2 of the paper. The rule is online, needs nothing beyond the current block, and is conservative in two ways: it bounds the actual perturbation $\|\mathbf{E}_{ik} \widetilde{\mathbf{L}}_{kk}^\top\|_F$ by the product of two norms, and it reserves the share $\tau m / n_B$ for every block whether or not the block needs it.

### Method B, the global budget

The global rule works with the sum itself. For a candidate format $p$ the realized contribution of block $(i, k)$ is
$$
d_{ik, p}^2 = 2\, \big\| \big[ \mathrm{fl}_p(\mathbf{A}_{ik}) - \mathbf{A}_{ik} \big] \widetilde{\mathbf{L}}_{kk}^\top \big\|_F^2 ,
$$
which costs one product of two $n_b \times n_b$ matrices per candidate, the same order as the triangular solve that produced the block. A running total $D^2$ of the contributions of the stored blocks is kept, and format $p$ is admissible for block $(i, k)$ only if
$$
D^2 + d_{ik, p}^2 \le \tau^2 m^2 .
$$
`fp64` always qualifies, since its contribution is zero under the arithmetic model. Because every accepted block keeps $D^2 \le \tau^2 m^2$, the matrix made of the blocks stored so far is always within $\tau m$ of $\bar{\boldsymbol{\Sigma}}$ in Frobenius norm, which is the only property the induction of the paper uses. The completion of every diagonal Cholesky and the positive definiteness of $\widetilde{\boldsymbol{\Sigma}}$ therefore hold for any rule that respects the running total, whatever order it visits the blocks in and however it spreads the budget. What the global rule changes is that a block may now carry a perturbation up to the whole of $\tau m$, instead of $\tau m / n_B$, when the other blocks are cheap, and that the cheapness of a block is measured by its actual perturbation.

Three ways of spending the global budget are compared.

**B-greedy, first feasible.** Blocks are visited in factorization order, and each takes the lowest format that fits in the budget left. It is the simplest online rule. Its weakness is that it spends the budget on whichever blocks come first, and the early block columns of a Cholesky factor carry the largest blocks. Once the budget is gone, even `fp32` is inadmissible for every remaining block, and the factor falls to `fp64`. At small $N$ this does not happen; at large $N$ it does, and the storage can exceed Method A's.

**B-paced, fair share with rollover.** Before block $(i, k)$ is stored, the budget left is divided by the number of blocks left, and the block takes the lowest format whose contribution is within that allowance. A block that uses less than its allowance leaves the difference to the blocks after it. The rule is online, costs nothing beyond the running total and a counter, and cannot be bankrupted by early blocks, because no block may take more than its share of what remains. It spends the budget evenly rather than where it buys the most, so it usually leaves part of the budget unused.

**B-optimal, offline allocation in two passes.** The first pass is a factorization under some rule, Method A here, that records for every block the contribution of every candidate format. The allocation then minimizes storage subject to the budget: every block starts in `fp32`, whose contributions are negligible, and the moves of each block from `fp32` to `bf16` and from `bf16` to `fp8` are taken in order of increasing extra contribution per byte saved until the budget is spent, a block's second move only after its first. When all blocks have the same size and the cost grows along the chain of formats, which it does since each step down loses bits, this is the minimum-storage assignment for the recorded costs. The second pass factors again and stores each block in its planned format, but it still checks the realized contribution against the running total, because the factor blocks depend on the formats stored before them and differ slightly between the passes; when the realized contribution would exceed the budget, the block falls back to the next format. The plan is limited to a fraction `PLAN_FRACTION` of the budget so that the second pass has room for these differences; without a margin the budget runs out in the last block columns and blocks there fall to `fp64`. The price of this variant is one extra factorization.

The guarantee of every variant assumes exact `fp64` arithmetic and storage, as in the paper. The code runs in finite-precision `fp64`, whose rounding is not covered by the bound; the dense verification at the end of the notebook checks the identity for $\|\boldsymbol{\Delta}\|_F$ and the smallest eigenvalue of $\widetilde{\boldsymbol{\Sigma}}$ directly when $N$ is small enough to form the matrix.

In [ ]:
TAU = 0.99                                                 # error-budget parameter
PLAN_FRACTION = 0.9                                        # share of the global budget the offline plan may use
N = 3 * len(keep) * len(GRID)
BLOCK_SIZE = 101 * max(3, int(np.ceil(3 * len(keep) / 144)))   # whole strides of one component per block, at most 144 block rows
SKIP_GREEDY = N * (N + 1) / 2 * 8 > 0.5 * GPU_GB * 1e9     # at large N the greedy variant falls to fp64, which would not fit
VERIFY_DENSE = N <= 12000                                  # L~ L~^T - Sigma is formed only when the dense matrices fit comfortably
fp32_factor_gb = (N * (N + 1) / 2 + N * BLOCK_SIZE / 2) * 4 / 1e9
print(f"N = {N:,}, block size {BLOCK_SIZE}, {int(np.ceil(N / BLOCK_SIZE))} block rows; factor stored in fp32 about {fp32_factor_gb:.1f} GB; "
      f"greedy variant {'skipped' if SKIP_GREEDY else 'included'}; dense verification {'on' if VERIFY_DENSE else 'off'}")
assert fp32_factor_gb < 0.75 * GPU_GB, f"the factor needs about {fp32_factor_gb:.0f} GB and this GPU has {GPU_GB:.0f} GB; reduce STRIDES or use the 80 GB A100"

CANDIDATES = {"fp8": torch.float8_e4m3fn, "bf16": torch.bfloat16, "fp32": torch.float32}      # lowest precision first
LARGEST = {"fp8": 448.0, "bf16": 3.39e38, "fp32": 3.40e38}                                    # largest finite number of each format
FORMAT_NAMES = ["fp8", "bf16", "fp32", "fp64"]
BYTES = {"fp8": 1, "bf16": 2, "fp32": 4, "fp64": 8}


def block_costs(A, Lkk):
    """Conversion error of A in every candidate format.

    Returns, per format, the Frobenius norm of the conversion error ||fl_p(A) - A||_F, used by the
    per-block rule, and the realized contribution d^2 = 2 ||(fl_p(A) - A) L_kk^T||_F^2 to the squared
    Frobenius norm of the perturbation of the reconstructed covariance matrix, used by the global rule.
    A format whose conversion overflows is marked infeasible. fp64 is the reference and costs nothing.
    """
    largest = float(A.abs().max())
    err, d2, finite = {}, {}, {}
    for name, dtype in CANDIDATES.items():
        E = A.to(dtype).to(torch.float64) - A
        finite[name] = largest <= LARGEST[name] and bool(torch.isfinite(E).all())
        err[name] = float(torch.linalg.norm(E)) if finite[name] else np.inf
        d2[name] = 2.0 * float(torch.linalg.norm(E @ Lkk.T))**2 if finite[name] else np.inf
    err["fp64"], d2["fp64"], finite["fp64"] = 0.0, 0.0, True
    return err, d2, finite

The four rules share one interface: given the conversion errors and the contributions of every candidate format for the current block, each returns the format to store. The offline plan is built by `plan_optimal` from the costs recorded in a first pass.

In [ ]:
class PerBlockRule:
    """Method A: the original rule, ||fl_p(A) - A||_F ||L_kk||_2 <= tau m / n_B."""
    name = "A: per-block budget"

    def __init__(self, m, nB, n_blocks):
        self.b = TAU * m / nB                                                 # the per-block share tau m / n_B

    def choose(self, i, k, err, d2, finite, norm_kk):
        for name in CANDIDATES:
            if finite[name] and err[name] * norm_kk <= self.b:
                return name
        return "fp64"


class GlobalGreedy:
    """Method B-greedy: first feasible format under the running global budget, blocks in factorization order."""
    name = "B-greedy: global budget"

    def __init__(self, m, nB, n_blocks):
        self.budget, self.used = (TAU * m)**2, 0.0

    def choose(self, i, k, err, d2, finite, norm_kk):
        for name in CANDIDATES:
            if finite[name] and self.used + d2[name] <= self.budget:
                self.used += d2[name]
                return name
        return "fp64"


class GlobalPaced:
    """Method B-paced: the lowest format whose contribution is within the budget left divided by the blocks left."""
    name = "B-paced: global budget"

    def __init__(self, m, nB, n_blocks):
        self.budget, self.used, self.left = (TAU * m)**2, 0.0, n_blocks

    def choose(self, i, k, err, d2, finite, norm_kk):
        allowance = (self.budget - self.used) / self.left
        self.left -= 1
        for name in CANDIDATES:
            if finite[name] and d2[name] <= allowance:
                self.used += d2[name]
                return name
        return "fp64"


class GlobalPlanned:
    """Method B-optimal, second pass: store the planned format when the running global budget allows it, else the next one."""
    name = "B-optimal: global budget"

    def __init__(self, m, nB, n_blocks, plan):
        self.budget, self.used, self.plan, self.fallbacks = (TAU * m)**2, 0.0, plan, 0

    def choose(self, i, k, err, d2, finite, norm_kk):
        start = FORMAT_NAMES.index(self.plan[i, k])
        for name in FORMAT_NAMES[start:]:
            if finite[name] and self.used + d2[name] <= self.budget:
                if name != self.plan[i, k]:
                    self.fallbacks += 1
                self.used += d2[name]
                return name
        return "fp64"


def plan_optimal(costs, sizes, budget):
    """Offline allocation from the recorded costs. Every block starts in fp32; the moves fp32 -> bf16 and bf16 -> fp8 of every
    block are then taken in order of increasing extra cost per byte saved until the budget is spent, each block's second move
    only after its first. With equal block sizes and costs that grow along the chain of formats this is the minimum-storage
    assignment for the recorded costs."""
    lower = [f for f in FORMAT_NAMES if f != "fp64"]                               # fp8, bf16, fp32
    plan, used = {}, 0.0
    for key in costs:
        plan[key] = "fp32" if np.isfinite(costs[key]["fp32"]) else "fp64"
        used += costs[key][plan[key]]
    if used > budget:                                                              # fp32 alone does not fit: demote the costliest fp32 blocks to fp64
        for key in sorted(costs, key=lambda key: -costs[key]["fp32"]):
            if used <= budget:
                break
            if plan[key] == "fp32":
                used -= costs[key]["fp32"]; plan[key] = "fp64"
    moves = []
    for key in costs:
        if plan[key] != "fp32":
            continue
        for higher, lower_f in zip(lower[::-1][:-1], lower[::-1][1:]):             # (fp32 -> bf16), (bf16 -> fp8)
            extra = costs[key][lower_f] - costs[key][higher]
            saved = sizes[key] * (BYTES[higher] - BYTES[lower_f])
            if np.isfinite(extra):
                moves.append((extra / saved, key, higher, lower_f, extra))
    for _, key, higher, lower_f, extra in sorted(moves, key=lambda t: t[0]):
        if plan[key] == higher and used + extra <= budget:
            used += extra; plan[key] = lower_f
    return plan, used


The factorization below is Algorithm 1 of the paper with the storage format of every off-diagonal block chosen by the rule passed in. It measures the conversion errors and the contributions of all candidate formats for every block, accumulates the realized contribution of the chosen format, and records the costs of every block when asked, for the offline plan. The block forward and back substitution solves with the stored factor, and the likelihood and the predictions reuse it.

In [ ]:
def mixed_precision_cholesky(model, rule, record=None):
    """Algorithm 1 of the paper with the storage format of every off-diagonal block chosen by `rule`.

    All arithmetic is fp64; the diagonal factor blocks are kept in fp64. For every off-diagonal block the conversion errors
    of all candidate formats are measured and passed to the rule, and the realized contribution d^2 of the chosen format is
    accumulated, so the Frobenius norm of L~ L~^T - Sigma is known exactly (under exact fp64 arithmetic) when the
    factorization ends. If `record` is a dict, the costs of every block in every format are stored in it.
    """
    blocks, nB = model.blocks, len(model.blocks)
    L, logdet, used = {}, 0.0, 0.0
    count = {name: 0 for name in FORMAT_NAMES}
    precision = np.full((nB, nB), np.nan)
    assignments, failure = [], None
    t0 = time.time()
    for k in range(nB):
        S = model.block(blocks[k], blocks[k])
        S.diagonal().add_(model.nugget[blocks[k]])
        row = [L[k, j].to(torch.float64) for j in range(k)]
        for Lkj in row:
            S.addmm_(Lkj, Lkj.T, alpha=-1.0)
        try:
            Lkk = torch.linalg.cholesky(S)
        except Exception as error:                                                 # not expected under the guarantee; reported if it happens
            failure = f"diagonal block {k}: {str(error).splitlines()[0]}"
            break
        L[k, k] = Lkk
        precision[k, k] = FORMAT_NAMES.index("fp64")
        logdet += 2.0 * float(torch.log(torch.diagonal(Lkk)).sum())
        norm_kk = float(torch.linalg.matrix_norm(Lkk, ord=2))
        for i in range(k + 1, nB):
            A = model.block(blocks[i], blocks[k])
            for j in range(k):
                A.addmm_(L[i, j].to(torch.float64), row[j].T, alpha=-1.0)
            A = torch.linalg.solve_triangular(Lkk.T, A, upper=True, left=False)
            err, d2, finite = block_costs(A, Lkk)
            if record is not None:
                record[i, k] = dict(d2)
            name = rule.choose(i, k, err, d2, finite, norm_kk)
            L[i, k] = A.to(CANDIDATES.get(name, torch.float64))
            used += d2[name]
            count[name] += 1
            precision[i, k] = FORMAT_NAMES.index(name)
            assignments.append({"i": i, "k": k, "format": name, "d2": d2[name], "d2_fp8": d2["fp8"], "d2_bf16": d2["bf16"], "d2_fp32": d2["fp32"],
                                "per_block_test_bf16": err["bf16"] * norm_kk, "per_block_test_fp32": err["fp32"] * norm_kk})
    elapsed = time.time() - t0
    stored_bytes = sum(block.nbytes for block in L.values())
    return dict(L=L, logdet=logdet, count=count, precision=precision, used=used, bytes=stored_bytes, time=elapsed,
                assignments=assignments, failure=failure)


def solve(L, blocks, rhs):
    # solve (L L^T) x = rhs by block forward and back substitution; rhs is a vector or a matrix with one right-hand side per column
    x, nB = rhs.clone().reshape(rhs.shape[0], -1), len(blocks)
    for k in range(nB):
        s = x[blocks[k]]
        for j in range(k):
            s -= L[k, j].to(torch.float64) @ x[blocks[j]]
        x[blocks[k]] = torch.linalg.solve_triangular(L[k, k], s, upper=False)
    for k in range(nB - 1, -1, -1):
        s = x[blocks[k]]
        for i in range(k + 1, nB):
            s -= L[i, k].to(torch.float64).T @ x[blocks[i]]
        x[blocks[k]] = torch.linalg.solve_triangular(L[k, k].T, s, upper=True)
    return x.reshape(rhs.shape)


def negative_log_likelihood(theta, model):
    # Gaussian log likelihood of the replicates, which share the covariance matrix; one factorization, one solve per replicate
    model.set_parameters(theta)
    L, logdet, count, precision = mixed_precision_cholesky(model)
    alpha = solve(L, model.blocks, model.y)                                        # (N, replicates)
    N, T = model.N, model.T
    sigma2 = float((model.y * alpha).sum()) / (N * T)                              # profiled variance of the first output
    stored_gb = sum(block.nbytes for block in L.values()) / 1e9
    return 0.5 * (N * T * np.log(sigma2) + T * logdet + N * T + N * T * np.log(2 * np.pi)), sigma2, count, stored_gb, precision



def negative_log_likelihood_from_factor(model, result):
    # the profiled Gaussian log likelihood of the stacked residuals, evaluated with a given factor
    alpha = solve(result["L"], model.blocks, model.y)
    N = model.N
    sigma2 = float(model.y @ alpha) / N
    return 0.5 * (N * np.log(sigma2) + result["logdet"] + N + N * np.log(2 * np.pi)), sigma2


def predict(model, L, F_new, sigma2, chunk=4):
    # predictive mean and standard deviation of the three standardized component curves of every new stride, in chunks of strides
    alpha = solve(L, model.blocks, model.y)
    all_rows = torch.arange(model.N, device=DEVICE)
    means, sds = [], []
    eta = torch.tensor(model.parameters["eta"], device=DEVICE)
    for a in range(0, F_new.shape[0], chunk):
        K_star = model.cross(F_new[a:a + chunk], all_rows)                                                 # (3 n_chunk 101, N)
        n_chunk = min(chunk, F_new.shape[0] - a)
        mean = (K_star @ alpha).reshape(3, n_chunk, model.T)
        V = solve(L, model.blocks, K_star.T)                                                                # (N, 3 n_chunk 101)
        explained = (K_star * V.T).sum(dim=1).reshape(3, n_chunk, model.T)
        prior = (torch.diagonal(model.amplitude) + eta)[:, None, None]
        var = sigma2 * (prior - explained)
        means.append(mean.cpu().numpy()); sds.append(np.sqrt(np.maximum(var.cpu().numpy(), 1e-12)))
    return np.concatenate(means, axis=1), np.concatenate(sds, axis=1)

In [ ]:
def dense_check(model, result):
    # forms L~ L~^T - Sigma, for verification only, and returns its Frobenius norm and the smallest eigenvalues
    blocks, nB = model.blocks, len(model.blocks)
    Sigma = torch.cat([torch.cat([model.block(blocks[i], blocks[j]) for j in range(nB)], dim=1) for i in range(nB)], dim=0)
    Sigma.diagonal().add_(model.nugget)
    Lt = torch.zeros_like(Sigma)
    for (i, k), block in result["L"].items():
        Lt[blocks[i], blocks[k]] = block.to(torch.float64)
    Delta = Lt @ Lt.T - Sigma
    return float(torch.linalg.norm(Delta)), float(torch.linalg.eigvalsh(Sigma)[0]), float(torch.linalg.eigvalsh(Lt @ Lt.T)[0])

### Size of the covariance matrix

In [ ]:
model = SpaceTimeTrivariateMatern(F_train, Z_train, GRID.astype(float), BLOCK_SIZE)
N, nB = model.N, len(model.blocks)
n_off = nB * (nB - 1) // 2
sizes = {(i, k): (model.blocks[i].stop - model.blocks[i].start) * (model.blocks[k].stop - model.blocks[k].start) for k in range(nB) for i in range(k + 1, nB)}
print(f"covariance matrix          {N:,} x {N:,}   (1 joint x {len(COMPONENTS)} components x {model.n} strides x {model.T} gait points)")
print(f"matrix in fp64             {N * N * 8 / 1e9:.2f} GB, Cholesky factor in fp64 {N * (N + 1) / 2 * 8 / 1e9:.2f} GB")
print(f"blocks                     {nB} x {nB} of up to {BLOCK_SIZE} points, {n_off} off-diagonal factor blocks")
print(f"GPU                        {GPU_NAME}, {GPU_GB:.1f} GB")

## Maximum likelihood estimation under each rule

The fourteen parameters are estimated by L-BFGS-B with forward-difference gradients, as in the earlier notebooks, with a mixed-precision factorization inside every evaluation of the likelihood, on the comparison strides, or on the `FIT_STRIDES` subset of them when that is set. The fit is done once for each rule: the per-block rule of Method A, then the greedy, the paced and the offline-optimal global rule, each with that rule's factorization inside the likelihood. For the offline rule every evaluation of the likelihood costs two factorizations, one to record the costs and one to store the plan. Each fit starts from the same values and is saved to its own checkpoint, so a rerun reads the checkpoints. With `FIT_EACH_RULE = False` only Method A is fitted and every rule is compared at its estimates.

Every rule keeps the reconstructed covariance matrix within $\tau m$ of $\bar{\boldsymbol{\Sigma}}$, so the four likelihood surfaces differ by amounts of the order of the perturbation and the four fits are expected to agree within the tolerance of the optimizer. The table of estimates below shows whether they do. The comparison that follows then runs each rule at its own estimates, so each column of the results is a complete pipeline, estimation and factorization, under one rule.

In [ ]:
MAX_ITERATIONS = 20
FIT_EACH_RULE = True                                       # fit the parameters under every rule; False fits Method A only
fit_model = SpaceTimeTrivariateMatern(F_fit, Z_fit, GRID.astype(float), 101 * max(3, int(np.ceil(3 * len(fit_rows) / 144))))
fit_nB = len(fit_model.blocks)

# starting values (fx, fy, fz): the univariate knee input and temporal ranges, unit variance ratios, 0.1 % nugget effects
l0 = np.full(3, {"hip": 0.354, "knee": 0.287, "ankle": 0.313}[JOINT])
lt0 = np.full(3, {"hip": 5.57, "knee": 4.34, "ankle": 4.46}[JOINT])
theta0 = np.concatenate([np.log(l0), np.log(lt0), [0.3, 0.3, 0.3], [0.0, 0.0], np.log([1e-3, 1e-3, 1e-3])])
bounds = [(np.log(0.05), np.log(5.0))] * 3 + [(np.log(1.0), np.log(30.0))] * 3 + [(-2.0, 2.0)] * 3 + [(-3.0, 3.0)] * 2 + [(np.log(1e-5), np.log(0.1))] * 3


def factor_under(rule_name, mdl, record=None):
    # one mixed-precision factorization of mdl under the named rule, at its current parameters
    nb_, m = len(mdl.blocks), mdl.m
    n_off_ = nb_ * (nb_ - 1) // 2
    if rule_name == "A: per-block budget":
        return mixed_precision_cholesky(mdl, PerBlockRule(m, nb_, n_off_), record)
    if rule_name == "B-greedy: global budget":
        return mixed_precision_cholesky(mdl, GlobalGreedy(m, nb_, n_off_), record)
    if rule_name == "B-paced: global budget":
        return mixed_precision_cholesky(mdl, GlobalPaced(m, nb_, n_off_), record)
    costs = {}
    first = mixed_precision_cholesky(mdl, PerBlockRule(m, nb_, n_off_), record=costs)             # first pass records the costs
    del first
    sizes_ = {(i, k): (mdl.blocks[i].stop - mdl.blocks[i].start) * (mdl.blocks[k].stop - mdl.blocks[k].start) for k in range(nb_) for i in range(k + 1, nb_)}
    plan, planned_used = plan_optimal(costs, sizes_, PLAN_FRACTION * (TAU * m)**2)
    rule = GlobalPlanned(m, nb_, n_off_, plan)
    r = mixed_precision_cholesky(mdl, rule, record)                                                # second pass stores the plan
    r["plan"] = {f: sum(1 for v in plan.values() if v == f) for f in FORMAT_NAMES}
    r["planned_used"], r["fallbacks"] = planned_used, rule.fallbacks
    return r


def fit_under(rule_name, verbose):
    # maximum likelihood estimation of the fourteen parameters with the named rule inside the likelihood
    tag = rule_name.split(":")[0].replace(" ", "")
    checkpoint = f"spacetime_checkpoint_{JOINT}_xyz_{fit_model.n}strides_{tag}.json"
    if os.path.exists(checkpoint):
        saved = json.load(open(checkpoint))
        print(f"{rule_name}: parameters from {checkpoint}")
        return np.array(saved["theta"]), saved["nll"], saved.get("evaluations", np.nan), saved.get("time_s", np.nan)
    best = {"nll": np.inf, "theta": None, "evaluations": 0}
    t_fit = time.time()

    def objective(th):
        if best["theta"] is not None and np.max(np.abs(th - best["theta"])) > 0.7:
            return best["nll"] + 1e6 * float(np.max(np.abs(th - best["theta"])))
        t0 = time.time()
        fit_model.set_parameters(th)
        r = factor_under(rule_name, fit_model)
        if r["failure"] is not None:
            return 1e30
        nll, sigma2 = negative_log_likelihood_from_factor(fit_model, r)
        best["evaluations"] += 1
        if nll < best["nll"]:
            best.update(nll=nll, theta=th.copy())
        if verbose or best["evaluations"] % 50 == 0:
            p = fit_model.parameters
            print(f"{tag:10s} evaluation {best['evaluations']:4d}  negative log likelihood {nll:13.2f}  blocks {'/'.join(FORMAT_NAMES)} {'/'.join(str(c) for c in r['count'].values())}  [{time.time() - t0:.1f} s]\n"
                  f"      input ranges {np.round(p['l'], 3).tolist()}  temporal ranges {np.round(p['lt'], 2).tolist()}  variances (standardized) {np.round(sigma2 * p['v'], 4).tolist()}  "
                  f"nugget effects (standardized) {[f'{x:.2e}' for x in sigma2 * p['eta']]}  cross-correlations fx-fy {p['rho'][0, 1]:.3f}, fx-fz {p['rho'][0, 2]:.3f}, fy-fz {p['rho'][1, 2]:.3f}", flush=True)
        return nll

    result = minimize(objective, theta0, method="L-BFGS-B", bounds=bounds, options={"maxiter": MAX_ITERATIONS, "eps": 1e-3})
    elapsed = time.time() - t_fit
    json.dump({"theta": best["theta"].tolist(), "nll": best["nll"], "evaluations": best["evaluations"], "time_s": elapsed}, open(checkpoint, "w"))
    print(f"{rule_name}: {result.message}, {result.nit} iterations, {best['evaluations']} likelihood evaluations, {elapsed / 60:.1f} min; checkpoint {checkpoint}", flush=True)
    return best["theta"], best["nll"], best["evaluations"], elapsed


RULES = ["A: per-block budget"] + ([] if SKIP_GREEDY else ["B-greedy: global budget"]) + ["B-paced: global budget", "B-optimal: global budget"]
fits = {}
for rule_name in RULES if FIT_EACH_RULE else RULES[:1]:
    th, nll, evaluations, elapsed = fit_under(rule_name, verbose=(rule_name == RULES[0]))
    fit_model.set_parameters(th)
    r = factor_under(rule_name, fit_model)
    _, sigma2 = negative_log_likelihood_from_factor(fit_model, r)
    fits[rule_name] = {"theta": th, "nll": nll, "evaluations": evaluations, "time_s": elapsed, "sigma2": sigma2, "count": r["count"], "p": fit_model.parameters}
    del r
for rule_name in RULES:
    fits.setdefault(rule_name, fits["A: per-block budget"])                          # without FIT_EACH_RULE every rule uses the estimates of Method A
del fit_model
gc.collect(); torch.cuda.empty_cache()

### Estimates under each rule

The estimates of the fourteen parameters, the profiled variance and the negative log likelihood under each rule, side by side, with the difference of every estimate from the estimate under Method A, the number of likelihood evaluations and the time of each fit, and the precision mix of the factor of the fit matrix at the estimates. The ranges are in standardized units of the inputs and in percent of the gait cycle, the variances and nugget effects in the standardized units of the residuals.

In [ ]:
rows_est = []
for name in RULES:
    f, p_ = fits[name], fits[name]["p"]
    rows_est.append({"rule": name, "negative log likelihood": f["nll"], "evaluations": f["evaluations"], "fit time [min]": f["time_s"] / 60,
                     **{f"input range {c}": p_["l"][j] for j, c in enumerate(COMPONENTS)}, **{f"temporal range {c}": p_["lt"][j] for j, c in enumerate(COMPONENTS)},
                     **{f"variance {c}": f["sigma2"] * p_["v"][j] for j, c in enumerate(COMPONENTS)}, **{f"nugget effect {c}": f["sigma2"] * p_["eta"][j] for j, c in enumerate(COMPONENTS)},
                     "cross-correlation fx-fy": p_["rho"][0, 1], "cross-correlation fx-fz": p_["rho"][0, 2], "cross-correlation fy-fz": p_["rho"][1, 2],
                     "blocks fp8 / bf16 / fp32 / fp64 at the estimates": " / ".join(str(f["count"][k]) for k in FORMAT_NAMES)})
estimates_by_rule = pd.DataFrame(rows_est).set_index("rule").T
estimates_by_rule.to_csv(f"estimates_by_rule_{JOINT}_xyz_{len(fit_rows)}strides.csv")
numeric = [r for r in estimates_by_rule.index if r not in ("blocks fp8 / bf16 / fp32 / fp64 at the estimates", "evaluations", "fit time [min]")]
difference = estimates_by_rule.loc[numeric].astype(float).sub(estimates_by_rule.loc[numeric, "A: per-block budget"].astype(float), axis=0)
print(f"fit strides {len(fit_rows)}, N = {3 * len(fit_rows) * 101:,}, {'every rule fitted' if FIT_EACH_RULE else 'Method A fitted, its estimates used for every rule'}")
display(estimates_by_rule)
if FIT_EACH_RULE:
    print("\ndifference from the estimates under Method A")
    display(difference.drop(columns=["A: per-block budget"]).rename(columns=lambda c: f"{c} minus A").style.format("{:+.5f}"))

## Comparison

Each rule is now run on the comparison matrix at its own estimates: the parameters are set, the matrix is factored under the rule, and the precision mix, the stored size of the factor in bytes with the real dimensions and data types of every stored block, the realized perturbation $\sqrt{\sum d^2}$ against its bound $\tau m$, the factorization time, the negative log likelihood and the predicted test curves are recorded. The methods run one after another and only one factor is kept on the GPU at a time. For the offline rule the first pass that records the costs is run at the rule's own parameters, and its time is reported separately. Differences between the columns therefore combine the effect of the rule on the estimates, shown in the table above, and the effect of the rule on the factorization at those estimates.

In [ ]:
def size(n_bytes):
    return f"{n_bytes / 1e9:.2f} GB" if n_bytes >= 1e9 else f"{n_bytes / 1e6:.2f} MB"

all_block_entries = sum(sizes.values()) + sum((model.blocks[k].stop - model.blocks[k].start)**2 for k in range(nB))
fp64_bytes = all_block_entries * 8                                                  # the same blocks, all stored in fp64 (diagonal blocks as full squares)
results, rows, baseline, all_assignments = {}, [], {}, []

def evaluate(name):
    # set the rule's own estimates, factor under the rule, score, and keep only what the report needs
    model.set_parameters(fits[name]["theta"])
    budget = (TAU * model.m)**2
    t0 = time.time()
    r = factor_under(name, model)
    overhead = time.time() - t0 - r["time"]                                         # the first pass of the offline rule
    row = {"method": name, "completed": r["failure"] is None, "failure": r["failure"] or "",
           **{f"blocks_{f}": r["count"][f] for f in FORMAT_NAMES}, **{f"share_{f}_%": 100 * r["count"][f] / n_off for f in FORMAT_NAMES},
           "stored_factor_MB": r["bytes"] / 1e6, "reduction_vs_fp64_%": 100 * (1 - r["bytes"] / fp64_bytes),
           "m": model.m, "tau_m": TAU * model.m, "global_error_sqrt_sum_d2": np.sqrt(r["used"]), "error_over_tau_m": np.sqrt(r["used"]) / (TAU * model.m),
           "budget_used_%": 100 * r["used"] / budget, "within_budget": r["used"] <= budget, "factorization_time_s": r["time"], "first_pass_time_s": overhead}
    if "plan" in r:
        row.update({f"planned_{f}": r["plan"][f] for f in FORMAT_NAMES}, planned_budget_used_pct=100 * r["planned_used"] / budget, fallbacks_in_pass2=r["fallbacks"])
    if r["failure"] is None:
        nll, sigma2 = negative_log_likelihood_from_factor(model, r)
        mean, sd = predict(model, r["L"], F_test, sigma2)
        if "pred_mean" not in baseline:
            baseline["pred_mean"], baseline["nll"], baseline["bytes"] = mean, nll, r["bytes"]
        row.update(negative_log_likelihood=nll, nll_minus_A=nll - baseline["nll"], reduction_vs_A_pct=100 * (1 - r["bytes"] / baseline["bytes"]),
                   max_prediction_difference_vs_A_BW=float(np.max(np.abs(scale[:, None, None] * (mean - baseline["pred_mean"])))))
        if VERIFY_DENSE:
            dn, lam_min, lam_min_tilde = dense_check(model, r)
            row.update(dense_frobenius_norm=dn, dense_over_tracked=dn / np.sqrt(r["used"]) if r["used"] > 0 else np.nan, lambda_min_Sigma=lam_min, lambda_min_Sigma_tilde=lam_min_tilde)
    results[name] = {k: r[k] for k in ("count", "precision", "used", "bytes", "time", "failure")}
    results[name]["budget"] = budget
    all_assignments.append(pd.DataFrame(r["assignments"]).assign(method=name))
    del r
    gc.collect(); torch.cuda.empty_cache()
    print(f"{name:26s} blocks {'/'.join(str(row[f'blocks_{f}']) for f in FORMAT_NAMES)}  stored {size(row['stored_factor_MB'] * 1e6)}  error {row['error_over_tau_m']:.4f} tau m  [{row['factorization_time_s']:.1f} s]", flush=True)
    return row

for name in RULES:
    rows.append(evaluate(name))
table = pd.DataFrame(rows).set_index("method")
table.to_csv(f"mixed_precision_budget_comparison_{JOINT}_xyz_{model.n}strides.csv")
pd.concat(all_assignments).to_csv(f"mixed_precision_budget_blocks_{JOINT}_xyz_{model.n}strides.csv", index=False)

### Results

Precision mix of the off-diagonal factor blocks, stored size of the factor, reduction relative to the same blocks in `fp64` and relative to Method A, the realized perturbation against its bound $\tau m$ at the rule's own $m$, the share of the global budget used, the factorization time, whether the factorization completed, the change in the negative log likelihood relative to Method A and the largest change in the predicted test curves. The row of B-optimal is followed by its plan, the share of the budget the plan used and the number of blocks the second pass had to demote.

In [ ]:
shown = pd.DataFrame({
    "blocks fp8 / bf16 / fp32 / fp64": [" / ".join(str(results[n]["count"][f]) for f in FORMAT_NAMES) for n in results],
    "fp8 share [%]": [f"{100 * results[n]['count']['fp8'] / n_off:.1f}" for n in results],
    "bf16 share [%]": [f"{100 * results[n]['count']['bf16'] / n_off:.1f}" for n in results],
    "stored factor": [size(results[n]["bytes"]) for n in results],
    "reduction vs fp64 [%]": [f"{100 * (1 - results[n]['bytes'] / fp64_bytes):.1f}" for n in results],
    "reduction vs A [%]": [f"{table.loc[n, 'reduction_vs_A_pct']:.1f}" for n in results],
    "error / (tau m)": [f"{np.sqrt(results[n]['used']) / table.loc[n, 'tau_m']:.4f}" for n in results],
    "budget used [%]": [f"{100 * results[n]['used'] / results[n]['budget']:.2f}" for n in results],
    "time [s]": [f"{results[n]['time']:.1f}" + (f" + {table.loc[n, 'first_pass_time_s']:.1f}" if table.loc[n, "first_pass_time_s"] > 0.5 else "") for n in results],
    "completed": ["yes" if results[n]["failure"] is None else "no" for n in results],
    "nll - nll(A)": [f"{table.loc[n, 'nll_minus_A']:+.4f}" for n in results],
    "max prediction change vs A [BW]": [f"{table.loc[n, 'max_prediction_difference_vs_A_BW']:.4f}" for n in results],
}, index=list(results))
print(f"fp64 factor would be {size(fp64_bytes)} as stored ({nB} full diagonal blocks and {n_off} off-diagonal blocks)")
r = table.loc["B-optimal: global budget"]
print(f"B-optimal plan: fp8 / bf16 / fp32 / fp64 = {' / '.join(str(int(r[f'planned_{f}'])) for f in FORMAT_NAMES)}, planned budget use {r['planned_budget_used_pct']:.2f} % "
      f"of a plan limited to {100 * PLAN_FRACTION:.0f} %, fallbacks in the second pass {int(r['fallbacks_in_pass2'])}; the first pass took {r['first_pass_time_s']:.1f} s")
shown

### Dense verification

For checking only, and only when $N$ is small enough to hold the matrix, which the setting of `VERIFY_DENSE` above decides, the cell forms $\widetilde{\boldsymbol{\Sigma}} - \bar{\boldsymbol{\Sigma}}$ from the stored factor and compares its Frobenius norm with the tracked $\sqrt{\sum d^2}$, which should agree to rounding, and reports the smallest eigenvalue of $\widetilde{\boldsymbol{\Sigma}}$ against that of $\bar{\boldsymbol{\Sigma}}$, which should stay above $(1 - \tau)\, \lambda_{\min}(\bar{\boldsymbol{\Sigma}})$.

In [ ]:
if VERIFY_DENSE:
    display(table[["dense_frobenius_norm", "global_error_sqrt_sum_d2", "dense_over_tracked", "lambda_min_Sigma", "lambda_min_Sigma_tilde"]].style.format("{:.4e}"))
a = all_assignments[0]
print(f"costs d^2 of the off-diagonal blocks in the run of Method A: fp8 median {a['d2_fp8'].median():.3e}, min {a['d2_fp8'].min():.3e}, max {a['d2_fp8'].max():.3e};  bf16 median {a['d2_bf16'].median():.3e}, min {a['d2_bf16'].min():.3e}, max {a['d2_bf16'].max():.3e};  "
      f"fp32 median {a['d2_fp32'].median():.3e}, max {a['d2_fp32'].max():.3e};  global budget of Method A {results['A: per-block budget']['budget']:.3e}")
print("The guarantee assumes exact fp64 arithmetic and storage; the implementation uses finite-precision fp64, whose rounding is not covered by the bound.")

### Precision assigned to each block

The storage format of every block of the Cholesky factor under each rule, in the layout of the per-block precision figure of Salvaña (2026) and with the same colors for all methods. The diagonal blocks are always `fp64`; the off-diagonal blocks are `fp8`, `bf16`, `fp32` or `fp64`. The percentages in each title refer to the off-diagonal blocks, and the title also gives the stored size and the realized perturbation in units of $\tau m$.

In [ ]:
PRECISION_COLORS = {"fp8": "#5E3C99", "bf16": "#1FBEC0", "fp32": "#F26F22", "fp64": "#4D4D4D"}     # purple / teal / orange / dark gray
cmap = ListedColormap([PRECISION_COLORS[f] for f in FORMAT_NAMES]); cmap.set_bad("white")
norm = BoundaryNorm(np.arange(len(FORMAT_NAMES) + 1) - 0.5, cmap.N)
names = list(results)
fig, axes = plt.subplots(1, len(names), figsize=(4.6 * len(names), 5.6), constrained_layout=True)
for ax, name in zip(axes, names):
    r = results[name]
    ax.imshow(np.ma.masked_invalid(r["precision"]), cmap=cmap, norm=norm, origin="upper", interpolation="nearest")
    ax.set_title(f"{name}\n" + "   ".join(f"{f} {100 * r['count'][f] / n_off:.0f}%" for f in FORMAT_NAMES) + f"\n{size(r['bytes'])}, error {np.sqrt(r['used'] / r['budget']):.2f} tau m", fontsize=10)
    ax.set_xlabel("block column $k$")
axes[0].set_ylabel("block row $i$")
fig.legend(handles=[Patch(facecolor=PRECISION_COLORS[f], label=f) for f in FORMAT_NAMES], loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.08))
fig.suptitle(f"Storage precision of the off-diagonal factor blocks under the per-block and the global error budget\n($N$ = {N:,} = 3 components $\\times$ {model.n} strides $\\times$ {model.T} gait points, $n_b$ = {BLOCK_SIZE}, {nB} $\\times$ {nB} blocks, $\\tau$ = {TAU}, diagonal blocks fp64)", fontsize=12, y=1.1)
fig.savefig(f"mixed_precision_budget_precision_maps_{JOINT}_xyz_{model.n}strides.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2), constrained_layout=True)
bottom = np.zeros(len(names))
for f in FORMAT_NAMES:
    share = np.array([100 * results[n]["count"][f] / n_off for n in names])
    ax.bar(range(len(names)), share, bottom=bottom, color=PRECISION_COLORS[f], label=f, width=0.6)
    for x, (s, b0) in enumerate(zip(share, bottom)):
        if s >= 4:
            ax.text(x, b0 + s / 2, f"{s:.0f}%", ha="center", va="center", fontsize=9, color="white")
    bottom += share
ax.set_xticks(range(len(names))); ax.set_xticklabels([n.replace(": ", "\n") for n in names], fontsize=9)
ax.set_ylabel("off-diagonal factor blocks [%]"); ax.set_ylim(0, 112); ax.set_yticks(range(0, 101, 20)); ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.18))
for x, n in enumerate(names):
    ax.text(x, 102, size(results[n]['bytes']), ha="center", va="bottom", fontsize=9)
ax.set_title("Precision mix of the stored Cholesky factor, with the stored size above each bar", loc="left", pad=10)
fig.savefig(f"mixed_precision_budget_precision_mix_{JOINT}_xyz_{model.n}strides.png", dpi=200, bbox_inches="tight")
plt.show()

### Reading the results

The table of estimates shows how far the four maximum likelihood fits agree. Because every rule keeps the reconstructed covariance within $\tau m$ of $\bar{\boldsymbol{\Sigma}}$, the likelihood surfaces differ by amounts of the order of the perturbation and the estimates should agree to the optimizer's tolerance; a larger difference would mean the perturbation is affecting the fit, which the bound does not permit. In the comparison, the per-block rule uses a small fraction of a percent of the global budget, because `fp32` blocks contribute almost nothing and the per-block share $\tau m / n_B$ shrinks with $n_B$. The global rules spend the same budget where it buys storage. The greedy variant shows the hazard of a global budget in an online setting: when the budget is exhausted early, every later block falls to `fp64`. The paced variant avoids this by construction, at the price of leaving budget unused. The offline allocation spends almost all of the budget on the cheapest blocks and gives the largest `bf16` share, at the price of one extra factorization and a safety margin. In every case the realized perturbation stays within $\tau m$, the smallest eigenvalue of the reconstructed covariance matrix stays above $(1 - \tau)$ times that of $\bar{\boldsymbol{\Sigma}}$, and the likelihood and the predictions change by amounts far below any practical significance. The size of the gain is set by how many blocks have a `bf16` perturbation below $\tau m$, which depends on the nugget effect: with relative nugget effects of order $10^{-4}$ the bulk of the factor needs `fp32` under either rule, and the global budget redistributes slack rather than changing the memory class of the problem.

## References

Apanasovich, T. V., Genton, M. G. and Sun, Y. (2012). A valid Matérn class of cross-covariance functions for multivariate random fields with any number of components. *Journal of the American Statistical Association*, 107, 180-193.

Gneiting, T., Kleiber, W. and Schlather, M. (2010). Matérn cross-covariance functions for multivariate random fields. *Journal of the American Statistical Association*, 105, 1167-1177.

Salvaña, M. L. O. (2026). Adaptive block mixed-precision Cholesky factorization for large-scale space-time Gaussian processes. In *Proceedings of the SC26 Workshops, HPC for Environmental and Earth Sciences (HPC4EES)*, Chicago, IL, USA, November 15-20, 2026.